In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter, binary_erosion, binary_dilation

from utils import *

In [2]:
folder = 'temp/'
files = sorted(glob.glob(folder + '*.fits'))
files

['temp/phi-fdt-flat_20250310T080009_V202607101034C_0563100100.fits',
 'temp/phi-fdt-flat_20260310T040003_V202607101039C_0663100100.fits',
 'temp/phi-fdt-fringes_20250310T080009_V202607101034C_0563100100.fits',
 'temp/phi-fdt-fringes_20260310T040003_V202607101039C_0663100100.fits',
 'temp/phi-fdt-ghost_20250310T080009_V202607101034C_0563100100.fits',
 'temp/phi-fdt-ghost_20260310T040003_V202607101039C_0663100100.fits']

In [5]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
flat_file, fringes_file, ghost_file = files[1::2]

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data

with fits.open(fringes_file) as hdul:
    fringes = hdul[0].data

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

ghost = demodulate(ghost)
fringes = demodulate(fringes)
flat_ = demodulate(flat)

In [5]:
plt.figure(figsize=(10,10))
plt.imshow(flat_[0], vmin=0.9, vmax=1.1)
plt.tight_layout()

In [6]:
plt.figure(figsize=(10,10))
plt.imshow(fringes[3], vmin=-1e-3, vmax=1e-3)
plt.tight_layout()

In [8]:
plt.figure(figsize=(10,10))
plt.imshow(flat_[2] - np.mean(flat_[2]), vmin=-5e-3, vmax=5e-3)
plt.tight_layout()

In [9]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T080009_V202503131733C_0563100100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T080608_V202503131733C_0563100125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T081208_V202503131835C_0563100150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T081808_V202503131935C_0563100175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T082408_V202503131935C_0563100200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T083008_V202503132033C_0563100225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/solo_L1_phi-fdt-ilam_20250310T083608_V202503141634C_0563100250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025

In [21]:
with fits.open(files[0]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

cpos = header['CONTPOS'] - 1
wv = read_wavelengths(header)
#data = data.reshape(6,4,2048,2048)[cpos]
data = calc_continuum(data, wv, continuum=cpos)

data -= dark * 0.4
data /= flat
data = realign(data)
data = demodulate(data)

xr, yr = reflection_point_predict(header)
reflection = reflect(gaussian_filter(data[0], 8), xr, yr)

In [11]:
plt.figure(figsize=(10,10))
plt.imshow(data[0], vmin=-200, vmax=200)
plt.tight_layout()

In [12]:
plt.figure(figsize=(10,10))
plt.imshow(data[0] - ghost[0] * reflection, vmin=-200, vmax=200)
plt.tight_layout()

In [7]:
i = 3

plt.figure(figsize=(10,10))
plt.imshow(data[i] - ghost[i] * reflection - fringes[i] * data[0], vmin=-30, vmax=30)
plt.tight_layout()

NameError: name 'data' is not defined

In [5]:
25 * (0.378 - 0.187) / np.pi

1.5199297065276007